# T9-bonus · Full pipeline

## Goal

Wire everything from `25` into `infra/pipelines`: `pac copilot pack` in
build (no auth), `powerplatform_solution`/`pac solution import` in deploy
(OIDC), `run_suite()` as the promotion gate, and quarantine as the
automated kill switch on gate failure. This is the notebook that makes the
curriculum an enterprise project, not a tutorial.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
for f in ["agent-ci.yml", "platform-ci.yml", "deploy.yml"]:
    assert Path(f"../infra/pipelines/{f}").exists(), f"missing {f}"


## Concept

Nothing here is new mechanism — it's `25` with a human removed from the
loop. The build/deploy split from finding #8 is what makes this safe to
automate: build needs no credentials at all, so a compromised build runner
can't touch a live environment; deploy needs SP credentials scoped to
exactly the target environment and gates on `run_suite()` before
`publish-changes` ever fires. Quarantine as the automated kill switch means
a failed gate doesn't just block promotion — it can actively pull an
already-promoted agent back if the gate is re-run post-deploy and fails.


## Build


In [ ]:
from pathlib import Path
print(Path("../infra/pipelines/agent-ci.yml").read_text())


In [ ]:
print(Path("../infra/pipelines/deploy.yml").read_text())


### Add the automated quarantine-on-gate-failure step


In [ ]:
deploy_yml_path = Path("../infra/pipelines/deploy.yml")
text = deploy_yml_path.read_text()
quarantine_step = '''
      - name: Quarantine on gate failure
        if: failure()
        run: |
          pac copilot quarantine --enable --name "${{ secrets.AGENT_SCHEMA_NAME }}" \
            --environment "${{ secrets.DATAVERSE_ENV_ID }}"
'''
if "Quarantine on gate failure" not in text:
    text += quarantine_step
    deploy_yml_path.write_text(text)
print("automated quarantine-on-failure step added to deploy.yml")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import yaml
with open("../infra/pipelines/deploy.yml") as f:
    doc = yaml.safe_load(f)
step_names = [s.get("name") for s in doc["jobs"]["deploy-and-gate"]["steps"]]
assert "Quarantine on gate failure" in step_names, "quarantine step missing from pipeline"
assert step_names.index("Run golden suite as the promotion gate") < step_names.index("Quarantine on gate failure")
print("pipeline shape verified: pack (no auth) -> import -> gate -> quarantine-on-failure")


## Cost


In [ ]:
print("No agent build/publish here — this notebook only edits pipeline YAML. Actual runs of this pipeline cost what 25's manual walkthrough already measured.")


## Teardown


In [ ]:
print("No teardown — this pipeline is the standing promotion path for the fleet from here on.")
